## Brands-1 işlem pipeline\n
\n
Bu notebook, `brands-1/` klasöründe yaptığımız tüm işlemleri adım adım (hücre hücre) tekrarlar.\n
\n
- Her adımın üstünde ne yaptığını anlatan kısa bir açıklama var.\n
- Çıktılar `brands-1/` içine yeni dosya isimleriyle yazılır (orijinalleri korur).\n

## 0) Kurulum ve yollar

Bu hücre: gerekli paketleri import eder ve `brands-1/` içindeki dosya yollarını tek yerde tanımlar.


In [ ]:
from pathlib import Path
import re
import pandas as pd

# Eğer çeviri hücresini çalıştıracaksanız:
# pip install deep-translator

BASE_DIR = Path.cwd() / "brands-1"

INPUT_RAW = BASE_DIR / "KeepaExport-2026-04-25-ProductFinder.csv"
OUT_STRIPPED = BASE_DIR / "KeepaExport-2026-04-25-ProductFinder_stripped.csv"
OUT_NONEMPTY = BASE_DIR / "KeepaExport-2026-04-25-ProductFinder_stripped_nonempty_noauthor.csv"
OUT_TR = BASE_DIR / "KeepaExport-2026-04-25-ProductFinder_stripped_nonempty_noauthor_tr.csv"
OUT_TR_CLEAN = BASE_DIR / "KeepaExport-2026-04-25-ProductFinder_stripped_nonempty_noauthor_tr_clean.csv"
OUT_TR_CLEAN_LEN20 = BASE_DIR / "KeepaExport-2026-04-25-ProductFinder_stripped_nonempty_noauthor_tr_clean_len20.csv"

BASE_DIR, INPUT_RAW.exists()

## 1) Gereksiz sütunları kaldır

Bu hücre: aşağıdaki sütunları kaldırıp yeni bir CSV üretir:
- `Sales Rank: Current`
- `Reviews: Rating`
- `Buy Box: Current`
- `Buy Box: Stock`
- `Amazon: Current`
- `Amazon: Stock`
- `New: Current`


In [ ]:
DROP_COLS = [
    "Sales Rank: Current",
    "Reviews: Rating",
    "Buy Box: Current",
    "Buy Box: Stock",
    "Amazon: Current",
    "Amazon: Stock",
    "New: Current",
]

df = pd.read_csv(INPUT_RAW, encoding="utf-8", quoting=1)
existing_drop = [c for c in DROP_COLS if c in df.columns]
df2 = df.drop(columns=existing_drop)
df2.to_csv(OUT_STRIPPED, index=False, encoding="utf-8")

print("Yazıldı:", OUT_STRIPPED)
print("Sütun:", len(df.columns), "->", len(df2.columns), "(çıkarılan:", len(existing_drop), ")")

## 2) `Author` sütununu sil + boş değer kontrolü (EAN hariç)

Bu hücre:
- `Author` sütunu varsa siler
- `Product Codes: EAN` sütununu **boş kontrolünden hariç** tutar
- Diğer sütunlardan herhangi biri boşsa satırı siler


In [ ]:
AUTHOR_COL = "Author"
EAN_COL = "Product Codes: EAN"

df = pd.read_csv(OUT_STRIPPED, encoding="utf-8", quoting=1)
if AUTHOR_COL in df.columns:
    df = df.drop(columns=[AUTHOR_COL])

check_cols = [c for c in df.columns if c != EAN_COL]
trimmed = df[check_cols].apply(lambda col: col.astype(str).str.strip())
mask_any_empty = df[check_cols].isna().any(axis=1) | trimmed.eq("").any(axis=1)

df2 = df.loc[~mask_any_empty].copy()
df2.to_csv(OUT_NONEMPTY, index=False, encoding="utf-8")

print("Yazıldı:", OUT_NONEMPTY)
print("Satır:", len(df), "->", len(df2), "(silinen:", int(mask_any_empty.sum()), ")")

## 3) Title + Categories: Tree Türkçe çeviri

Bu hücre `deep-translator` ile Almanca `Title` ve `Categories: Tree` alanlarını Türkçe’ye çevirir ve iki yeni sütuna yazar:
- `Title (TR)`
- `Categories: Tree (TR)`

Not: Bu adım API kullandığı için zaman alabilir.


In [ ]:
from deep_translator import GoogleTranslator
import time
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeoutError

COL_TITLE = "Title"
COL_CAT = "Categories: Tree"
COL_TITLE_TR = "Title (TR)"
COL_CAT_TR = "Categories: Tree (TR)"

SAVE_EVERY = 50
DELAY_SECONDS = 0.25
TRANSLATE_TIMEOUT = 5
RATE_LIMIT_WAIT = 60
MAX_RETRIES = 4
MAX_CHARS = 5000


def is_rate_limit_error(e) -> bool:
    if e is None:
        return False
    msg = str(e).lower()
    return any(k in msg for k in ["429", "rate", "limit", "too many", "quota", "blocked"])


def _translate_one(translator, text: str):
    def _do():
        return translator.translate(text)

    with ThreadPoolExecutor(max_workers=1) as ex:
        fut = ex.submit(_do)
        try:
            return fut.result(timeout=TRANSLATE_TIMEOUT), True, None
        except (FuturesTimeoutError, Exception) as e:
            return "", False, e


def translate_with_retry(translator, text: object):
    if pd.isna(text) or str(text).strip() == "":
        return "", True
    s = str(text).strip()[:MAX_CHARS]
    for attempt in range(MAX_RETRIES):
        out, ok, err = _translate_one(translator, s)
        if ok:
            return out, True
        if err and is_rate_limit_error(err) and attempt < MAX_RETRIES - 1:
            print(f"\n[Rate limit] {RATE_LIMIT_WAIT} saniye bekleniyor (deneme {attempt + 1}/{MAX_RETRIES})...")
            time.sleep(RATE_LIMIT_WAIT)
            continue
        return "", False
    return "", False


df = pd.read_csv(OUT_NONEMPTY, encoding="utf-8", quoting=1)
if COL_TITLE_TR not in df.columns:
    df[COL_TITLE_TR] = ""
if COL_CAT_TR not in df.columns:
    df[COL_CAT_TR] = ""

translator = GoogleTranslator(source="de", target="tr")

n = len(df)
for i in range(n):
    # resume-friendly: ikisi de doluysa geç
    if str(df.at[i, COL_TITLE_TR]).strip() and str(df.at[i, COL_CAT_TR]).strip():
        continue

    title_tr, ok1 = translate_with_retry(translator, df.at[i, COL_TITLE])
    time.sleep(DELAY_SECONDS)
    cat_tr, ok2 = translate_with_retry(translator, df.at[i, COL_CAT])
    time.sleep(DELAY_SECONDS)

    df.at[i, COL_TITLE_TR] = title_tr
    df.at[i, COL_CAT_TR] = cat_tr

    if (i + 1) % SAVE_EVERY == 0:
        df.to_csv(OUT_TR, index=False, encoding="utf-8")
        print(f">>> Saved {i + 1}/{n}")

df.to_csv(OUT_TR, index=False, encoding="utf-8")
print("Yazıldı:", OUT_TR)

## 4) Fiyatı TL’ye çevir + Renewed/Yenilenmiş temizliği

Bu hücre:
- `List Price: Current` değerini 55 ile çarpıp `TL fiyat` sütununu ekler.
- `Title (TR)` içinde geçen `renewed / yenilenmiş` ifadelerini başlıktan siler.


In [ ]:
COL_PRICE = "List Price: Current"
COL_TL = "TL fiyat"
MULTIPLIER = 55

_PRICE_RE = re.compile(r"([0-9]+(?:[.,][0-9]+)?)")

_RENEWED_PATTERNS = [
    re.compile(r"\s*[\(\[]\s*amazon\s+renewed\s*[\)\]]\s*", re.IGNORECASE),
    re.compile(r"\s*[\(\[]\s*renewed\s*[\)\]]\s*", re.IGNORECASE),
    re.compile(r"\s*[\(\[]\s*yenilenmiş\s*[\)\]]\s*", re.IGNORECASE),
    re.compile(r"\s*[\(\[]\s*yenilenmis\s*[\)\]]\s*", re.IGNORECASE),
    re.compile(r"(\s+|^)(amazon\s+renewed|renewed|yenilenmiş|yenilenmis)(\s+|$)", re.IGNORECASE),
]


def parse_price(val):
    if pd.isna(val):
        return None
    s = str(val).strip()
    if not s:
        return None
    m = _PRICE_RE.search(s)
    if not m:
        return None
    num = m.group(1).replace(",", ".")
    try:
        return float(num)
    except ValueError:
        return None


def clean_renewed(text):
    if pd.isna(text):
        return text
    s = str(text)
    for rx in _RENEWED_PATTERNS:
        s = rx.sub(" ", s)
    s = re.sub(r"\s{2,}", " ", s).strip()
    s = re.sub(r"^\s*[-–—]\s*", "", s).strip()
    s = re.sub(r"\s*[-–—]\s*$", "", s).strip()
    return s


df = pd.read_csv(OUT_TR, encoding="utf-8", quoting=1)

# TL fiyat
eur = df[COL_PRICE].apply(parse_price)
df[COL_TL] = (pd.to_numeric(eur, errors="coerce") * MULTIPLIER).round(2)
cols = list(df.columns)
cols.remove(COL_TL)
cols.insert(cols.index(COL_PRICE) + 1, COL_TL)
df = df[cols]

# Başlık temizliği (Title (TR))
before = df["Title (TR)"].fillna("").astype(str)
df["Title (TR)"] = df["Title (TR)"].apply(clean_renewed)
after = df["Title (TR)"].fillna("").astype(str)
print("Değişen başlık sayısı:", int((before != after).sum()))

df.to_csv(OUT_TR_CLEAN, index=False, encoding="utf-8")
print("Yazıldı:", OUT_TR_CLEAN)

## 5) Türkçe ürün adı kısa olanları sil

Bu hücre: `Title (TR)` uzunluğu 20 karakterden kısa olan satırları siler.


In [ ]:
MIN_LEN = 20
COL = "Title (TR)"

df = pd.read_csv(OUT_TR_CLEAN, encoding="utf-8", quoting=1)
lens = df[COL].fillna("").astype(str).str.strip().str.len()
mask_remove = lens < MIN_LEN

df2 = df.loc[~mask_remove].copy()
df2.to_csv(OUT_TR_CLEAN_LEN20, index=False, encoding="utf-8")

print("Önce:", len(df), "Silinen:", int(mask_remove.sum()), "Sonra:", len(df2))
print("Yazıldı:", OUT_TR_CLEAN_LEN20)

## 6) Son CSV’yi Excel’e çevir

Bu hücre: pipeline’ın ürettiği son dosyayı (`*_len20.csv`) `.xlsx` formatına çevirir.


In [ ]:
# pip install openpyxl
import pandas as pd

EXCEL_OUT = BASE_DIR / "KeepaExport-2026-04-25-ProductFinder_final_len20.xlsx"

df = pd.read_csv(OUT_TR_CLEAN_LEN20, encoding="utf-8", quoting=1)
df.to_excel(EXCEL_OUT, index=False, engine="openpyxl")
print("Yazıldı:", EXCEL_OUT)